In [1]:
# Import the necessary Python modules.

# Data Management/Investigation
import pandas as pd
import requests # For downloading the website
from bs4 import BeautifulSoup # For parsing the website
import numpy as np
import missingno as miss

# Plotting libraries
from plotnine import *
import matplotlib.pyplot as plt

# For iterating through directory
import os

# Misc
import warnings
# Targeted suppression: the OSHPD workbooks trip harmless openpyxl warnings
# ("Cannot parse header or footer", "Print area cannot be set...") on nearly
# every file. Suppress just those; everything else still surfaces.
warnings.filterwarnings("ignore", module = "openpyxl")


United States Census Bureau’s “Explore Census Data”<br/>
Identify income in the past twelve months by ZIP Code

In [2]:
# Load the Census Data collected from the U.S. Census Bureau.
census_data = pd.read_csv("Census_Data/ACSST5Y2019.S1901_data_with_overlays_2021-11-04T111610.csv", 
                          low_memory = False)

In [3]:
# Look at the datatypes of the Census Data.
census_data.dtypes

GEO_ID            str
NAME              str
S1901_C01_001E    str
S1901_C01_001M    str
S1901_C01_002E    str
                 ... 
S1901_C04_014M    str
S1901_C04_015E    str
S1901_C04_015M    str
S1901_C04_016E    str
S1901_C04_016M    str
Length: 130, dtype: object

In [4]:
# All ZIP Codes in our data.
all_zip_codes = pd.DataFrame((census_data
                              .NAME[1:]
                              .str
                              .replace("ZCTA5 ", "")
                              .unique()
                              .astype("str")
                             ), columns = ["zip_code"])

In [5]:
# Total number of ZIP Codes in CA in our data.
# https://data.ca.gov/dataset/county-and-zip-code-references
ca_zip_codes = pd.read_csv("Census_Data/zip-code-list.csv")

In [6]:
# Convert to object data type.
ca_zip_codes = ca_zip_codes.astype("str")

In [7]:
# Number of zip codes in CA in our data.
len(all_zip_codes
    .merge(ca_zip_codes, how = "inner", on = "zip_code")
    .zip_code
    .unique()
   )

1763

American Hospital Directory (AHD) - List of Hospital Websites and Additional Information

In [8]:
def scrap_AHD():
    """
    Web scraping techniques to collect a list of hospitals in California and push the results into a CSV to be saved.
    
    Arguments:
        None
    
    Return:
        None
    """
    # Prepare list to hold the UN information.
    scraped_data = []

    # AHD website - CA search results for all hospitals.
    url = "https://www.ahd.com/list_cms.php?mstate%5B%5D=CA&listing=1&viewmap=0"

    # Download the webpage.
    page = requests.get(url)

    # If a connection was successfully reached.
    if page.status_code == 200:
        # Parse the webpage.
        soup = BeautifulSoup(page.content, "html.parser")

        # Identify all table rows on the webpage.
        # Ignore the first seven rows.
        table_rows = soup.find_all("tr")[7:485]

        # Iterate through each table row.
        for table_row in table_rows:
            # Identify all cells within the table row.
            table_cell = table_row.find_all("td")

            # Hospital Name.
            name = str(table_cell[0].text)

            # Hospital Beds.
            beds = int(table_cell[1].text)

            # Hospital City.
            city = str(table_cell[2].text)

            # Pull the items in the cell.
            html_soup = BeautifulSoup(str(table_cell))

            # Iterate through the cell and find all a-tags with href.
            for href_url in html_soup.find_all("a", href = True):
                # Pull the href from the table for only the first entry.
                website = "https://www.ahd.com" + href_url["href"]
                break

            # Add row to dataframe.
            scraped_data.append([name, beds, city, website])

    # Convert the holding list to Pandas DataFrame.
    dat = pd.DataFrame(scraped_data, columns = ["name", "beds", "city", "website"])

    # Save the hospital information to a CSV.
    dat.to_csv("Hospital_Data/AHD_list.csv")

In [9]:
# Load the AHD list we scrapped.
ahd_list = pd.read_csv("Hospital_Data/AHD_list.csv")

# Print out the results of scrapping.
print("Number of Hosps with Beds:", len(ahd_list.query("beds > 0")))
print("Number of Hosps with No Beds:", len(ahd_list.query("beds == 0")))

Number of Hosps with Beds: 408
Number of Hosps with No Beds: 70


In [10]:
def read_OSHPD(base_code):
    """
    Reads OSHPD files where files are housed in separate folders for each hospital.
    Looks for pricing data given a HCPCS code and returns results for all hospitals in a Pandas DataFrame.

    Arguments:
        HCPCS code of interest to collect pricing data

    Return:
        Pandas DataFrame
    """

    # Collect matching rows per file, then concatenate once at the end.
    matches = []

    # Directory we will read from to obtain pricing data.
    directory = "Hospital_Data/OSHPD_2019"

    # Files that could not be parsed, reported at the end instead of silently dropped.
    skipped = []

    # Iterate through each folder in the directory.
    for subdir, dirs, files in os.walk(directory):
        # Iterate through each file in the folder.
        for file in files:
            # If the file is an Excel file.
            # (Note: this deliberately preserves the original case-sensitive check,
            # which skips the handful of .XLSX/.xlsm files, so results match the
            # committed CSVs. See README.)
            if file[-4:] == "xlsx" or file[-3:] == "xls":
                # Determine the file path of the Excel doc.
                file_path = os.path.join(subdir, file)

                try:
                    # Read every sheet of the Excel file to a dict of DataFrames.
                    sheet_to_df_map = pd.read_excel(file_path, sheet_name = None)

                    # Iterate through each sheet in the file.
                    for sheet in sheet_to_df_map.values():
                        # Make sheet into dataframe.
                        df = pd.DataFrame(sheet)

                        # Check for base_code as a string in any column.
                        entry_base_code_str = df[df.eq(str(base_code)).any(axis = 1)]

                        # If the string version is found.
                        if entry_base_code_str.shape[0] > 0:
                            # Add rows (tagged with the hospital directory) to the results.
                            entry = entry_base_code_str.copy()
                            entry["HospitalDirectory"] = subdir
                            matches.append(entry)

                            # Continue to next sheet.
                            continue

                        # Check base_code as an integer in any column.
                        entry_base_code_int = df[df.eq(int(base_code)).any(axis = 1)]

                        # If the int version is found.
                        if entry_base_code_int.shape[0] > 0:
                            # Add rows (tagged with the hospital directory) to the results.
                            entry = entry_base_code_int.copy()
                            entry["HospitalDirectory"] = subdir
                            matches.append(entry)
                except Exception as e:
                    # Unreadable file (e.g. the encrypted workbooks in the OSHPD
                    # download) or a sheet that cannot be matched: record it and
                    # move on -- but never silently, unlike the original bare except.
                    skipped.append((file_path, f"{type(e).__name__}: {e}"))
                    continue

    # Report any skipped files loudly rather than hiding them.
    if skipped:
        print(f"WARNING: skipped {len(skipped)} unreadable file(s):")
        for path, reason in skipped:
            print(f"  - {path} ({reason})")

    # Concatenate all matches into the final dataframe.
    final_dt = pd.concat(matches, sort = False) if matches else pd.DataFrame(columns = ["HospitalDirectory"])

    # A run that finds nothing at all indicates a broken environment, not an
    # empty market -- fail loudly instead of writing an empty CSV downstream.
    assert final_dt.shape[0] > 0, "read_OSHPD found no rows in any file -- check the Excel engines and pandas version."

    # Return the final dataframe.
    return final_dt

In [11]:
# Collect hospital pricing data from OSHPD directory of hospital files.
oshpd_70450 = read_OSHPD(70450)

WARNING *** file size (2719680) not 512 + multiple of sector size (512)


WARNING *** file size (16320) not 512 + multiple of sector size (512)


  - Hospital_Data/OSHPD_2019/Clovis Commuity Medical Center/106100005_CDM_All_2019.xlsx (ValueError: Unable to read workbook: could not read stylesheet from Hospital_Data/OSHPD_2019/Clovis Commuity Medical Center/106100005_CDM_All_2019.xlsx.
This is most probably because the workbook source files contain some invalid XML.
Please see the exception for more details.)
  - Hospital_Data/OSHPD_2019/Madera Community Hospital/106201281_PCT_CHG_2019.xls (XLRDError: Workbook is encrypted)
  - Hospital_Data/OSHPD_2019/Community Regional Medical Center - Fresno/106100717_CDM_All_2019.xlsx (ValueError: Unable to read workbook: could not read stylesheet from Hospital_Data/OSHPD_2019/Community Regional Medical Center - Fresno/106100717_CDM_All_2019.xlsx.
This is most probably because the workbook source files contain some invalid XML.
Please see the exception for more details.)


In [12]:
# Select only the columns of interest to identify the correct pricing.
oshpd_70450_clean = oshpd_70450[["HospitalDirectory", "Unnamed: 1", "Unnamed: 2", "PRICE", 
                     "June 2019 Prices", "Charge Amount", "STD AMOUNT",
                     "Rate", "Amount", "Price", "UNIT CHARGE AMOUNT",
                     "AMOUNT", "Total Price ", "CUR CHG"]]

In [13]:
# Remove spaces in column names.
oshpd_70450_clean.columns = oshpd_70450_clean.columns.map(lambda x: x.replace(' ', '_'))

# Remove colons in column names.
oshpd_70450_clean.columns = oshpd_70450_clean.columns.map(lambda x: x.replace(':', ''))

In [14]:
# Coalesce the pricing columns: for each row, take the first non-NA value
# scanning left to right across the candidate price columns (everything after
# HospitalDirectory and Unnamed_1). This is equivalent to the original
# right-to-left fillna cascade, but explicit and safe under pandas copy-on-write.
coalesced_price = oshpd_70450_clean.iloc[:, 2:].bfill(axis = 1).iloc[:, 0]

In [15]:
# Create new column named "HospitalPrice_70450" from the coalesced price.
oshpd_70450_clean = oshpd_70450_clean.copy()
oshpd_70450_clean["HospitalPrice_70450"] = coalesced_price

In [16]:
# Collect final cleaned OSHPD data.
oshpd_70450_final = oshpd_70450_clean[["HospitalDirectory", "HospitalPrice_70450"]].copy()

In [17]:
# Convert all results to numeric where all strings will be converted to NAs.
oshpd_70450_final["HospitalPrice_70450"] = pd.to_numeric(oshpd_70450_final.HospitalPrice_70450, errors = "coerce")

In [18]:
# Drop the duplicates for each hospital. Only keep the first entry.
oshpd_70450_final = oshpd_70450_final.drop_duplicates(subset = ["HospitalDirectory"], keep = "first")

In [19]:
# If the price looks like the HCPCS Code 70450, then mark as NA.
oshpd_70450_final.loc[oshpd_70450_final["HospitalPrice_70450"] == 70450, "HospitalPrice_70450"] = np.nan

In [20]:
# Rename hospital directory as hospital name without path.
oshpd_70450_final["HospitalName"] = oshpd_70450_final["HospitalDirectory"].str.split("/").str[-1]

In [21]:
# Save the OSHPD data to a CSV.
oshpd_70450_final.filter(items = ["HospitalName", "HospitalPrice_70450"]).to_csv("Hospital_Data/OSHPD_list.csv")

In [22]:
# Reload the OSHPD pricing data for hospitals in CA with Zip Code.
oshpd_70450_zipcode = pd.read_csv("Hospital_Data/OSHPD_ZipCode_list.csv")

In [23]:
oshpd_70450_zipcode["ZipCode"] = oshpd_70450_zipcode["ZipCode"].astype(str)

In [24]:
# Remove the first row since it was a subheader.
census_data_clean = census_data.iloc[1:, :].copy()

In [25]:
# Clean up the Zip Code entries.
census_data_clean["NAME"] = census_data_clean["NAME"].str.replace("ZCTA5 ", "").astype(str)

In [26]:
# Merge the hospital pricing data with census data.
merged_data = pd.merge(oshpd_70450_zipcode, census_data_clean, how = "left", left_on = "ZipCode", right_on = "NAME")

In [27]:
# Collect the columns of interest we will include in our machine learning pipeline.
final_data = merged_data[["HospitalName", "HospitalPrice_70450", "ZipCode", "S1901_C01_012E", "S1901_C01_013E"]]

In [28]:
# Rname columns to understand what they actually are.
final_data = final_data.rename(columns = {"S1901_C01_012E" : "MedIncome", "S1901_C01_013E" : "MeanIncome"})

In [29]:
# Save the final CSV final to push into our data analysis.
final_data.to_csv("Hospital_Data/Final_Pricing.csv")